# ♟️ Python Chess — Engine, AI, and Interactive Board

A from-scratch 8×8 chess implementation in pure Python, playable in this notebook.

**What's inside**
- **`chess_engine.py`** — board representation, move generation for all six piece types, castling, en passant, promotion, and full game-end detection (checkmate, stalemate, threefold repetition, 50-move rule, insufficient material).
- **`chess_ai.py`** — minimax with alpha-beta pruning and piece-square-table evaluation.
- **`chess_ui.py`** — an interactive `ipywidgets` board you can click to play.
- **`test_chess.py`** — 14 unit tests including `perft` correctness checks.

**Run this notebook top-to-bottom.** Cell 4 gives you the interactive game.

## 1. Install dependencies (if needed)

Only `ipywidgets` is required. Uncomment and run if you don't have it.

In [1]:
# !pip install ipywidgets

In [9]:
import os
os.chdir(os.path.expanduser("~/Desktop/chess1"))
print("Now working in:", os.getcwd())
print("Files:", os.listdir())

Now working in: /Users/godfredantwikoduah/Desktop/chess1
Files: ['chess_engine.py', 'chess.ipynb', '__pycache__', 'README.md', 'test_chess.py', '.ipynb_checkpoints', 'chess_ui.py', 'chess_ai.py']


## 2. Quick engine sanity check

Print the starting position and count legal moves (should be 20).

In [10]:
from chess_engine import Board, Color, Move, PieceType

board = Board()
print(board.ascii())
print()
print(f"Side to move:       {board.turn.name}")
print(f"Legal moves:        {len(board.legal_moves())}   (expected: 20)")
print(f"In check:           {board.is_in_check(board.turn)}")
print(f"Game result so far: {board.game_result()}")

8 ♜ ♞ ♝ ♛ ♚ ♝ ♞ ♜
7 ♟ ♟ ♟ ♟ ♟ ♟ ♟ ♟
6 . . . . . . . .
5 . . . . . . . .
4 . . . . . . . .
3 . . . . . . . .
2 ♙ ♙ ♙ ♙ ♙ ♙ ♙ ♙
1 ♖ ♘ ♗ ♕ ♔ ♗ ♘ ♖
  a b c d e f g h

Side to move:       WHITE
Legal moves:        20   (expected: 20)
In check:           False
Game result so far: None


## 3. Correctness: `perft` move-generation test

`perft(n)` counts every leaf in the move tree at depth `n`. The reference values
from [chessprogramming.org](https://www.chessprogramming.org/Perft_Results) are
widely used to verify chess engines — any discrepancy points to a bug in
castling, en-passant, pins, or promotion logic.

In [11]:
import time
from chess_engine import perft

reference = {1: 20, 2: 400, 3: 8902}
for d, expected in reference.items():
    t0 = time.perf_counter()
    got = perft(Board(), d)
    dt = time.perf_counter() - t0
    mark = "✓" if got == expected else "✗"
    print(f"  {mark}  perft({d}) = {got:>5d}   (expected {expected:>5d})   {dt*1000:>6.1f} ms")

  ✓  perft(1) =    20   (expected    20)      0.8 ms
  ✓  perft(2) =   400   (expected   400)      7.5 ms
  ✓  perft(3) =  8902   (expected  8902)     64.0 ms


## 4. Play against the AI

Click a piece to select it; legal destinations turn green (empty) or red (captures).
Click a destination to move. The AI plays Black.

- **AI depth** slider: 1 = fast/weak, 4 = slow/strong. Depth 3 is the sweet spot in pure Python (~0.1–1s per move).
- **Play other color**: swap sides (AI plays White).
- **New game**: reset the position.

Pawn promotion is automatic to Queen (the right choice in the overwhelming majority of positions).

In [12]:
from chess_ui import ChessUI

game = ChessUI(ai_depth=3)
game.display()

## 5. Inspect what the AI is doing

Get the AI's chosen move plus search statistics — useful for understanding how
alpha-beta pruning cuts down the search space.

In [6]:
from chess_ai import get_best_move, evaluate

b = Board()
# Play a few opening moves by hand to get an interesting middlegame-like position.
for uci in ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4"]:
    from_sq = (8 - int(uci[1]), "abcdefgh".index(uci[0]))
    to_sq   = (8 - int(uci[3]), "abcdefgh".index(uci[2]))
    b.make_move(Move(from_sq, to_sq))

print(b.ascii())
print()

for depth in (1, 2, 3, 4):
    t0 = time.perf_counter()
    move, info = get_best_move(b, depth=depth, return_stats=True)
    dt = time.perf_counter() - t0
    print(f"depth={depth}   best move: {move}   score: {info['score_cp']:+5d} cp"
          f"   nodes: {info['nodes']:>6d}   time: {dt*1000:6.0f} ms")

8 ♜ . ♝ ♛ ♚ ♝ ♞ ♜
7 ♟ ♟ ♟ ♟ . ♟ ♟ ♟
6 . . ♞ . . . . .
5 . . . . ♟ . . .
4 . . ♗ . ♙ . . .
3 . . . . . ♘ . .
2 ♙ ♙ ♙ ♙ . ♙ ♙ ♙
1 ♖ ♘ ♗ ♕ ♔ . . ♖
  a b c d e f g h

depth=1   best move: g8f6   score:   +30 cp   nodes:     32   time:     20 ms
depth=2   best move: g8f6   score:  -100 cp   nodes:    255   time:     49 ms
depth=3   best move: d8f6   score:  +145 cp   nodes:   2332   time:    386 ms
depth=4   best move: g8f6   score:  -110 cp   nodes:  12940   time:   1512 ms


## 6. Engine vs engine

Watch the engine play itself. Useful for spotting evaluation weaknesses or
testing changes to the AI.

In [7]:
import time
from chess_engine import Board
from chess_ai import get_best_move

b = Board()
max_plies = 40
t0 = time.perf_counter()
for ply in range(max_plies):
    if b.game_result() is not None:
        break
    move = get_best_move(b, depth=2)   # depth 2 so self-play finishes quickly
    b.make_move(move)

print(b.ascii())
print()
print(f"Plies played: {len(b.position_history) - 1}")
print(f"Result:       {b.game_result() or '(ongoing — hit ply cap)'}")
print(f"Total time:   {time.perf_counter() - t0:.1f}s")

8 . ♚ . . ♜ ♝ . ♜
7 ♟ ♟ ♟ ♛ ♟ ♟ ♟ ♟
6 . . ♞ ♟ ♝ ♞ . .
5 . . . . . . . .
4 . . . . . . . .
3 ♙ . ♘ ♙ ♙ ♘ . .
2 . ♙ ♙ ♗ ♗ ♙ ♙ ♙
1 ♖ . . ♕ ♖ . ♔ .
  a b c d e f g h

Plies played: 25
Result:       Draw by threefold repetition
Total time:   0.7s


## 7. Run the test suite

All 14 tests should pass.

In [8]:
!python test_chess.py 2>&1 | tail -20

test_cannot_castle_through_check (__main__.BasicRuleTests.test_cannot_castle_through_check) ... ok
test_castling_kingside (__main__.BasicRuleTests.test_castling_kingside) ... ok
test_en_passant_capture (__main__.BasicRuleTests.test_en_passant_capture) ... ok
test_initial_position_has_20_legal_moves (__main__.BasicRuleTests.test_initial_position_has_20_legal_moves) ... ok
test_initial_turn_is_white (__main__.BasicRuleTests.test_initial_turn_is_white) ... ok
test_pawn_promotion_generates_four_moves (__main__.BasicRuleTests.test_pawn_promotion_generates_four_moves) ... ok
test_pinned_piece_cannot_move_off_the_pin (__main__.BasicRuleTests.test_pinned_piece_cannot_move_off_the_pin) ... ok
test_fools_mate (__main__.GameEndTests.test_fools_mate) ... ok
test_insufficient_material_k_plus_knight_vs_k (__main__.GameEndTests.test_insufficient_material_k_plus_knight_vs_k) ... ok
test_insufficient_material_kk (__main__.GameEndTests.test_insufficient_material_kk) ... ok
test_stalemate (__main__.GameE

## Extending this project

Good next steps if you want to sharpen it further for a quant portfolio:

1. **Stronger search** — quiescence search (extends captures at leaves to avoid
   the horizon effect), iterative deepening, transposition tables (Zobrist hashing).
2. **Better evaluation** — mobility, king safety, pawn structure (doubled,
   isolated, passed pawns), tempo.
3. **Bitboard representation** — replace the 2D array with 64-bit integers per
   piece type. Much faster move generation; the same data structure shows up in
   HFT order-book work.
4. **UCI protocol** — let the engine talk to any chess GUI (Arena, CuteChess)
   and play against [Stockfish](https://stockfishchess.org/) at fixed depth.
5. **Self-play data + ML eval** — generate training data for a small neural
   network evaluator (NNUE-style). This bridges classical search with ML and
   is exactly the kind of thing quant interviewers like to probe.